## Optional Lab : Loss and Cost Functions

In [1]:
import deeplearning_ai.machine_learning_specialization.course1.week1.models.*;
import deeplearning_ai.machine_learning_specialization.course1.week1.*;

import java.util.function.Function;
import static java.lang.System.out;

//NumPy Like dependency : 
import org.nd4j.linalg.api.ndarray.INDArray;
import org.nd4j.linalg.factory.Nd4j;

In [2]:
House house1 = new House(1.f, 300.f);
House house2 = new House(2.f, 500.f);
House house3 = new House(1.5f,350.f); 

House[] houses = new House[] {house1, house2, house3};      //<--- Labeled Datas


//************************************************************/
//BUT we will not use above. We will seperate house's fields : 
//************************************************************/
float[] housesAreas = new float[] {house1.sqftArea, house2.sqftArea, house3.sqftArea};
float[] housesPrices = new float[] {house1.price, house2.price, house3.price};

INDArray areaFeatures = Nd4j.create(housesAreas);
INDArray priceTargets = Nd4j.create(housesPrices);

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


### Goal : 
We want to give **a unknown house area** to model (or function here), function **should return possible price**. <br>

We have a model (= function): <br> 
```java
float singleVariablePredictFunction(float houseArea) = weight * housearea + bias;
```
or 
```java 
INDArray prices = singleVariablePredictFunction(INDArray houseAreas) = [weight * iThHouseArea + bias];
```

In [3]:
//Let function's weigth = w, bias = b be.
float w = 100.f;
float b = 100.f;

SingleVariablePredictFunction singleVariablePredictFunction = SingleVariablePredictFunctionImpl.create(w,b);

//f == singleVariablePredictFunction = area -> w * area + b == 100 * area + 100 

> ***weight (w)*** and ***bias (b)*** also called ***model's parameters***

Are our model's predictions success? (***With $w=100, b=100$ parameters***)

In [4]:
INDArray predictions = singleVariablePredictFunction.apply(areaFeatures);
out.println("Predictions  : " + predictions);      //Predictions  : [  200.0000,  300.0000,  250.0000]
out.println("Real targets : " + priceTargets);     //Real targets : [  300.0000,  500.0000,  350.0000]

Predictions  : [  200.0000,  300.0000,  250.0000]
Real targets : [  300.0000,  500.0000,  350.0000]


### Loss (L) Function 

Our model's $ meanError = \frac{|300-200|+|500-300|+|350-250|}{3} $

In [5]:
MeanError calculateMeanError = (predicts, reals) -> {
    int m = (int) predicts.length();
    float meanError = 0;
    for (int iTh = 0; iTh < m; iTh++)
        meanError += Math.abs(predicts.getFloat(iTh) - reals.getFloat(iTh));    
        //iThHouseError = | predict_i - real_i |  <-- We are measuring to error by 'Euclidean Distance'
    return meanError /= m;
};

float meanError = calculateMeanError.apply(predictions, priceTargets);
out.println("Abs mean error = " + meanError);   //Abs mean error = 133.33333

Abs mean error = 133.33333


> But we ***will not use*** error function above. Because we want to ***update model's parameters looking at model's error*** (with derivative) ***for model's better predictions***. 

So we should choose loss function for easy derivative compute. <br>

Maybe lossFunctionForIThSample = realTarget_i - modelPredict_i. <br>
- ***But lossFunction always should not been negative***. (negative error is not logical)
- And we want to easy derivative calculate it.

We know 'all real number's squared is positive'. ***Let we define $ lossFunctionForIThSample = (realTarget_i - modelPredict_i)^{2} $. <br>
The function also easier derivative compute than previous (because polynomial function)*** : 

In [6]:
float lossFuncForSingleSample(float modelPredict, float realValue) {
    return (float) Math.pow(modelPredict - realValue, 2);
}


//First sample loss:
float loss1 = lossFuncForSingleSample(predictions.getFloat(0), priceTargets.getFloat(0));
out.println("First sample (house) predict loss = " + loss1);    //First sample (house) predict loss = 10000.0

First sample (house) predict loss = 10000.0


In [7]:
LossFunction computeAllSampleLoss = (predictions,realValues) -> {
    int m = (int) predictions.length();
    float[] losses = new float[m];
    for (int i = 0; i < m; i++)
        losses[i] = lossFuncForSingleSample(predictions.getFloat(i), realValues.getFloat(i));

    return Nd4j.create(losses);
};

INDArray allLosses = computeAllSampleLoss.apply(predictions, priceTargets);
out.println("All house price losses : " + allLosses);   
//All house price losses : [       1e4,       4e4,       1e4]     //<-- 1e4 == 1*(10^4)

All house price losses : [       1e4,       4e4,       1e4]


### Cost (J) Function

In [8]:
//We have All Losses array for f(x) = 100 * x + 100 model : [       1e4,       4e4,       1e4]
// x is house's area. 

CostFunction meanSquaredError = (predictions, realValues) -> {
    INDArray losses = computeAllSampleLoss.apply(predictions, realValues);
    int m = (int) losses.length();      //<--- m = sampleSize
    float mse = 0.f;
    for (int iTh = 0; iTh < m; iTh++)
        mse += losses.getFloat(iTh);
    return mse /= m;
};

In [9]:
float mseForWeightIs100BiasIs100Model = meanSquaredError.calculate(predictions, priceTargets);
out.println("Our Mean Squared Error (MSE) = " + mseForWeightIs100BiasIs100Model);
//Our Mean Squared Error (MSE) = 20000.0     //= 2e4 

Our Mean Squared Error (MSE) = 20000.0


**We can define** cost (J) function : <br>

$$ 
J = meanSquaredError \\
J = (\frac{1}{sampleSize}) \cdot \sum_{i = 0}^{sampleSize} (lossFuncForSingleSample^{(i)}(predict^{(i)}, real^{(i)})) \\
$$ 
Math notation : 
$$
J = (\frac{1}{m}) \cdot \sum_{i = 0}^{m} (L^{(i)}(\hat{y}^{(i)}, y^{(i)})) \\
$$
$ \hat{y}^{(i)} $ = model ($ f_{w,b} = f $)'s **i.** sample prediction. So, <br>
$$
J = (\frac{1}{m}) \cdot \sum_{i = 0}^{m} (L^{(i)}(f({x}^{(i)}), y^{(i)})) \\
$$

#### Andrew Ng's Cost Function Definition : 

$$
J_{ng} = \frac{1}{2} \cdot J_{us}
$$

Both are correct. But we will use Ng's definition in these notebooks. Because notebooks has been designed by Ng's courses. <br>
When we come calculate derivatives, I will share $J_{us}$'s derivative formulas. 
$$
J_{default} = J_{ng} = (\frac{1}{2 \cdot m}) \cdot \sum_{i = 0}^{m} (L^{(i)}(f({x}^{(i)}), y^{(i)})) \\
$$

In [10]:
CostFunction J = (predictions, realValues) -> (1.f / 2) * meanSquaredError.calculate(predictions, realValues);

## Different Parameters, Different Costs

We chosed parameters : $ w = 100, b = 100 $. 

In [11]:
float cost = J.calculate(predictions, priceTargets);
out.println("J = J(w,b) = J(" + w +  "," + b + ") = " + cost);    //J = J(w,b) = J(100.0,100.0) = 10000.0

J = J(w,b) = J(100.0,100.0) = 10000.0


In [12]:
void updateModelsParametersAndPrintCost(float newWeight, float newBias, SingleVariablePredictFunction singleVariablePredictFunction) {
    singleVariablePredictFunction.updateWeight(newWeight);
    singleVariablePredictFunction.updateBias(newBias);

    INDArray predictions = singleVariablePredictFunction.apply(areaFeatures);
    float cost = J.calculate(predictions, priceTargets);
    out.println("J = J(w,b) = J(" + newWeight +  "," + newBias + ") = " + cost);
}

Let choose $ w = 200, b = 100 $ :

In [13]:
w = 200.f;
b = 100.f;
updateModelsParametersAndPrintCost(w, b, singleVariablePredictFunction)     //J = J(w,b) = J(200.0,100.0) = 416.66666

J = J(w,b) = J(200.0,100.0) = 416.66666


Let choose $ w = 250, b = 150 $ :

In [14]:
w = 250.f;
b = 150.f;
updateModelsParametersAndPrintCost(w, b, singleVariablePredictFunction)     //J = J(w,b) = J(250.0,150.0) = 10520.833

J = J(w,b) = J(250.0,150.0) = 10520.833


If Model, <br> 
$ f(area) = 100 * area + 100 \Rightarrow J(w,b) = J(100,100) = 10000 $, <br>
$ f(area) = 200 * area + 100 \Rightarrow J(w,b) = J(200,100) = 416 $ <br>
$ f(area) = 250 * area + 150 \Rightarrow J(w,b) = J(200,100) = 10520 $ <br>

Question : <br>
***How we choose best parameters for minimum error ?***